# 01 — Exploratory data analysis

Distributions and data-quality checks on the Instacart warehouse.
Run `scripts/build_warehouse.py` first.

In [ ]:
import duckdb
import matplotlib.pyplot as plt

con = duckdb.connect('../data/warehouse.duckdb', read_only=True)
plt.rcParams['figure.figsize'] = (9, 4)

## Scale of the data

In [ ]:
for t in ['aisles', 'departments', 'products', 'orders',
            'order_products__prior', 'order_products__train']:
    n = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'{t:24s} {n:>12,}')

## Data-quality gates

- duplicate order_ids?
- orders with zero products?
- impossible hours / days?

In [ ]:
checks = {
    'dup order_ids': "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM orders",
    'orders w/o products': '''SELECT COUNT(*) FROM orders o
        LEFT JOIN order_products__prior p ON p.order_id = o.order_id
        WHERE o.eval_set='prior' AND p.order_id IS NULL''',
    'bad hour': 'SELECT COUNT(*) FROM orders WHERE order_hour_of_day NOT BETWEEN 0 AND 23',
    'bad dow': 'SELECT COUNT(*) FROM orders WHERE order_dow NOT BETWEEN 0 AND 6',
}
for name, q in checks.items():
    print(f'{name:20s} {con.execute(q).fetchone()[0]}')

## Basket size distribution

In [ ]:
basket = con.execute('''
    SELECT order_id, COUNT(*) AS n
    FROM order_products__prior GROUP BY 1
''').df()
print(basket['n'].describe([.5, .9, .99]).round(1))
basket['n'].clip(upper=50).hist(bins=50)
plt.title('Basket size (clipped at 50)'); plt.xlabel('products'); plt.show()

## Reorder rate by department (top 10)

In [ ]:
dept = con.execute('''
    SELECT d.department, AVG(op.reordered) AS r
    FROM order_products__prior op
    JOIN products p ON p.product_id = op.product_id
    JOIN departments d ON d.department_id = p.department_id
    GROUP BY 1 ORDER BY 2 DESC LIMIT 10
''').df()
dept.plot.barh(x='department', y='r', legend=False)
plt.title('Reorder rate by department'); plt.xlabel('reorder rate'); plt.show()